In [1]:
# =========================================================
# SigLIP2 - Linear Probe (Backbone Frozen)
# =========================================================
import os, sys, re, random, math
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from tqdm.auto import tqdm
from pathlib import Path
from collections import Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# -------------------- Config --------------------
# MODEL_ID = "qihoo360/fg-clip2-so400m"            # FG-CLIP2
MODEL_ID = "google/siglip2-giant-opt-patch16-384"  # SigLIP2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
BATCH = 64
EPOCHS = 5
LR = 1e-3
NUM_WORKERS = 2
PRINT_ALL_TEST = True

# Early Stopping
EARLY_STOP_PATIENCE  = 5
EARLY_STOP_MIN_DELTA = 0.002

# Paths
train_dir = "/kaggle/input/v1-masakan-nusantara/action.v10i.folder/train"
test_dir  = "/kaggle/input/data-mining-action-2025/test/test/"
OUTPUT_CSV = "submission_linear_probe.csv"

# MASK 
USE_MASK   = True
MASK_ROOT  = "/kaggle/input/mask-633-orient-auto-resize640-ref-contrast-adapt/train"
MASK_MODE  = "dual"        # "crop" | "multiply" | "dual"
MASK_PAD   = 0.06
MASK_MIN_PIX = 64
MASK_EXTS  = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

# pHash fallback
PHASH_HASH_SIZE = 8
PHASH_MAX_DIST  = 8   # toleran

# Debug
DEBUG_MASK_SAMPLES = 5

# Label target
labels_ind = [
    "Ayam Bakar","Ayam Betutu","Ayam Goreng","Ayam Pop","Bakso",
    "Coto Makassar","Gado Gado","Gudeg","Nasi Goreng","Pempek",
    "Rawon","Rendang","Sate Madura","Sate Padang","Soto"
]

# ====== Mask-class config ======
# Cara membinarisasi semantic mask:
#  - "any"      : semua non-zero dianggap foreground (seperti kode awal)
#  - "target"   : hanya piksel kelas yang sesuai label gambar (label_id+1)
#  - "majority" : pilih kelas non-zero dengan area terbesar di mask
MASK_BIN_STRATEGY = "target"   # "any" | "target" | "majority"

# Tabel nilai piksel -> nama kelas (sesuai daftar)
MASK_CLASS_TABLE = {
    0: "background",
    1: "Ayam Bakar", 2: "Ayam Betutu", 3: "Ayam Goreng", 4: "Ayam Pop",
    5: "Bakso", 6: "Coto Makassar", 7: "Gado Gado", 8: "Gudeg",
    9: "Nasi Goreng", 10: "Pempek", 11: "Rawon", 12: "Rendang",
    13: "Sate Madura", 14: "Sate Padang", 15: "Soto"
}
MASK_ALLOWED_VALUES = set(MASK_CLASS_TABLE.keys())

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

print("=== RUNTIME ===")
print("Python :", sys.version.split()[0])
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    try: print("GPU    :", torch.cuda.get_device_name(0))
    except Exception: pass
print(f"EarlyStopping: patience={EARLY_STOP_PATIENCE}, min_delta={EARLY_STOP_MIN_DELTA}")

# -------------------- Load backbone --------------------
from transformers import AutoImageProcessor, AutoModel
print(f"\n=== MODEL ===\nLoading model: {MODEL_ID}")
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE)
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
for p in model.parameters(): p.requires_grad = False
model.eval()

def call_processor(img):
    kwargs = dict(images=img, return_tensors="pt")
    if "fg-clip2" in MODEL_ID.lower():
        kwargs["max_num_patches"] = 196   # aman utk FG-CLIP2, diabaikan SigLIP2
    return image_processor(**kwargs)

@torch.no_grad()
def backbone_features(pixel_values_dict):
    if isinstance(pixel_values_dict, dict):
        pixel_values_dict = {k: v.to(DEVICE) for k, v in pixel_values_dict.items()}
    else:
        pixel_values_dict = {"pixel_values": pixel_values_dict.to(DEVICE)}
    if DEVICE == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            if hasattr(model, "get_image_features"):
                feats = model.get_image_features(**pixel_values_dict)
            elif hasattr(model, "encode_image"):
                feats = model.encode_image(**pixel_values_dict)
            else:
                out = model(**pixel_values_dict); feats=None
                for k in ["image_embeds","pooler_output","last_hidden_state"]:
                    if isinstance(out, dict) and k in out:
                        feats = out[k]
                        if k=="last_hidden_state": feats=feats[:,0]
                        break
                if feats is None: raise RuntimeError("Tidak menemukan field fitur gambar.")
    else:
        if hasattr(model, "get_image_features"):
            feats = model.get_image_features(**pixel_values_dict)
        elif hasattr(model, "encode_image"):
            feats = model.encode_image(**pixel_values_dict)
        else:
            out = model(**pixel_values_dict); feats=None
            for k in ["image_embeds","pooler_output","last_hidden_state"]:
                if isinstance(out, dict) and k in out:
                    feats = out[k]
                    if k=="last_hidden_state": feats=feats[:,0]
                    break
            if feats is None: raise RuntimeError("Tidak menemukan field fitur gambar.")
    if feats.dim()==3: feats=feats[:,0]
    return feats

# -------------------- Dataset & Mask pairing --------------------
IMG_SIZE = 384

# Augment: kurang agresif jika crop
train_tfms_crop_safe = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.05)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
])
train_tfms_default = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
])
valid_tfms = transforms.Compose([ transforms.Resize((IMG_SIZE, IMG_SIZE)) ])

IMAGE_EXTS = (".jpg",".jpeg",".png",".bmp",".webp")

def build_from_subfolders(root):
    paths, labs = [], []
    for cls in sorted(os.listdir(root)):
        cdir = Path(root)/cls
        if not cdir.is_dir(): continue
        for p in cdir.rglob("*"):
            if p.suffix.lower() in IMAGE_EXTS:
                paths.append(str(p)); labs.append(cls)
    return paths, labs

# --- Mask helpers (regex robust) ---
def is_mask_filename(name: str) -> bool:
    """
    File mask bila stem diakhiri 'mask' (+opsional 'semantic' / 'seg'),
    dengan pemisah _, -, atau spasi.
    """
    stem = Path(name).stem.lower()
    return bool(re.search(r'(^|[_\-\s])mask([_\-\s]?(?:semantic|seg))?$', stem))

def strip_mask_suffix(stem: str) -> str:
    """Hilangkan sufiks ...(_|-)mask[(_|-)semantic|seg] di akhir stem."""
    s = stem.lower()
    s = re.sub(r'([_\-\s])?mask([_\-\s]?(?:semantic|seg))?$', '', s)
    return s

def build_mask_stem_index(mask_root: str, prefer_exts=(".png",)):
    """STEM (tanpa sufiks mask) -> path mask; hanya file yang bertanda mask."""
    idx = {}
    if not (mask_root and os.path.isdir(mask_root)):
        print(f"[WARN] MASK_ROOT invalid: {mask_root}")
        return idx
    scanned=kept=over=0; sample=[]
    for p in Path(mask_root).rglob("*"):
        if not p.is_file(): 
            continue
        if p.suffix.lower() not in MASK_EXTS:
            continue
        scanned += 1
        if not is_mask_filename(p.name):  # skip RGB di folder mask
            continue
        key = strip_mask_suffix(Path(p).stem)
        if key in idx:
            if (Path(idx[key]).suffix.lower() not in prefer_exts) and (p.suffix.lower() in prefer_exts):
                idx[key] = str(p); over += 1
        else:
            idx[key] = str(p); kept += 1
            if len(sample) < 5: sample.append(os.path.relpath(p, mask_root))
    print(f"Mask scan   : scanned={scanned}, indexed(mask-only)={kept}, overwritten_by_prefer={over}")
    if sample:
        print("Sample mask indexed:"); [print("  -", s) for s in sample]
    return idx

# --- dHash (pHash sederhana) ---
def compute_dhash(img: Image.Image, hash_size=8) -> int:
    g = img.convert("L").resize((hash_size+1, hash_size), Image.BILINEAR)
    arr = np.asarray(g, dtype=np.int16)
    diff = arr[:, 1:] > arr[:, :-1]
    bits = 0
    for v in diff.flatten():
        bits = (bits << 1) | int(v)
    return bits

def hamming(a: int, b: int) -> int:
    return (a ^ b).bit_count()

def build_mask_rgb_phash(mask_root: str, hash_size=PHASH_HASH_SIZE):
    """Kumpulkan RGB (bukan mask) dari MASK_ROOT + dHash-nya."""
    items = []
    if not (mask_root and os.path.isdir(mask_root)):
        return items
    sample=[]
    for p in Path(mask_root).rglob("*"):
        if not p.is_file(): continue
        if p.suffix.lower() not in IMAGE_EXTS: continue
        if is_mask_filename(p.name):  # skip file mask
            continue
        try:
            with Image.open(p) as im:
                dh = compute_dhash(im, hash_size)
            items.append((str(p), Path(p).stem.lower(), dh))
            if len(sample) < 5: sample.append(os.path.relpath(p, mask_root))
        except Exception:
            continue
    print(f"Mask RGB phash: {len(items)} candidates")
    if sample:
        print("Sample RGB in MASK_ROOT:"); [print("  -", s) for s in sample]
    return items

class MaskPairer:
    """Cari mask utk gambar train: (1) exact STEM, (2) phash terhadap RGB di MASK_ROOT."""
    def __init__(self, mask_root, phash_hash_size=8, phash_max_dist=8):
        self.mask_root = mask_root
        self.stem2mask = build_mask_stem_index(mask_root)
        self.rgb_phash  = build_mask_rgb_phash(mask_root, hash_size=phash_hash_size)
        self.phash_max_dist = phash_max_dist
        self.cache = {}

    def find_for_train_rgb(self, fp_img: str, base_root: str):
        if fp_img in self.cache: 
            return self.cache[fp_img], "cache"
        # exact STEM
        try: rel = os.path.relpath(fp_img, base_root)
        except Exception: rel = fp_img
        stem = strip_mask_suffix(Path(rel).stem.lower())
        mp = self.stem2mask.get(stem, None)
        if mp:
            self.cache[fp_img] = mp
            return mp, "stem"
        # fallback phash
        try:
            with Image.open(fp_img) as im:
                qh = compute_dhash(im, PHASH_HASH_SIZE)
        except Exception:
            self.cache[fp_img] = None
            return None, "error"
        best = (999, None, None)  # (dist, rgb_path, rgb_stem)
        for rgb_path, rgb_stem, dh in self.rgb_phash:
            d = hamming(qh, dh)
            if d < best[0]:
                best = (d, rgb_path, rgb_stem)
                if d == 0: break
        if best[0] <= self.phash_max_dist:
            mp = self.stem2mask.get(strip_mask_suffix(best[2]), None)
            self.cache[fp_img] = mp
            return mp, f"phash@{best[0]}"
        self.cache[fp_img] = None
        return None, "miss"

    def coverage_report(self, items, base_root, max_show=5):
        exact=ph=miss=0; miss_ex=[]
        for fp in items:
            mp, how = self.find_for_train_rgb(fp, base_root)
            if mp and how.startswith("stem"): exact += 1
            elif mp and how.startswith("phash"): ph += 1
            else:
                miss += 1
                if len(miss_ex) < max_show:
                    miss_ex.append(os.path.relpath(fp, base_root))
        total = len(items)
        cov = 100.0 * (exact+ph) / max(1,total)
        print(f"Mask pairing  : exact={exact}, phash={ph} (<= {self.phash_max_dist}), miss={miss} / total={total} → coverage={cov:.1f}%")
        if miss_ex:
            print("Contoh tanpa pasangan (maks 5):")
            for s in miss_ex: print(" -", s)

def print_mask_pair_samples(items, base_root, pairer, k=5):
    if not pairer: 
        print("PAIR: (mask disabled)")
        return
    cnt = 0
    for fp in items:
        mp, how = pairer.find_for_train_rgb(fp, base_root)
        if mp:
            print("PAIR:", os.path.relpath(fp, base_root), "->", os.path.relpath(mp, pairer.mask_root), f"({how})")
            cnt += 1
            if cnt >= k: break
    if cnt == 0:
        print("PAIR: (belum ada pasangan terdeteksi untuk sampel)")

def check_mask_label_consistency(paths, labels, base_root, pairer, max_show=10):
    mismatches = []
    for fp, y in zip(paths, labels):
        mp, _ = pairer.find_for_train_rgb(fp, base_root)
        if not mp: continue
        m = np.asarray(Image.open(mp).convert("L"))
        vals = m[m > 0]
        if vals.size == 0: continue
        u, c = np.unique(vals, return_counts=True)
        maj_val = int(u[np.argmax(c)])   # 1..15
        expected_val = int(y) + 1
        if maj_val != expected_val:
            mismatches.append((
                os.path.relpath(fp, base_root),
                MASK_CLASS_TABLE.get(maj_val, f"{maj_val}"),
                MASK_CLASS_TABLE.get(expected_val, f"{expected_val}")
            ))
            if len(mismatches) >= max_show: break
    if mismatches:
        print("\n[WARN] Mismatch folder label vs mask majority (contoh):")
        for rel, maj, exp in mismatches:
            print(f" - {rel} | mask→{maj} vs label→{exp}")
    else:
        print("\nMask ↔ label: konsisten (berdasarkan majority class).")

# ---------- PRINT CONFIG ----------
print("\n=== DATA CONFIG ===")
print(f"TRAIN_DIR : {train_dir}")
print(f"TEST_DIR  : {test_dir}")
print(f"USE_MASK  : {USE_MASK}")
print(f"MASK_ROOT : {MASK_ROOT}")
print(f"MASK_MODE : {MASK_MODE} | PAD={MASK_PAD} | MIN_PIX={MASK_MIN_PIX}")
print(f"PHASH     : size={PHASH_HASH_SIZE}, max_dist={PHASH_MAX_DIST}")
print(f"MASK_BIN  : {MASK_BIN_STRATEGY} (target uses label_id+1)")

train_paths, train_labels = build_from_subfolders(train_dir)
print("Class distribution:", Counter(train_labels))

# ---------- Labels enc ----------
le = LabelEncoder(); le.fit(labels_ind)
unknown = sorted(set(train_labels) - set(labels_ind))
if unknown:
    raise ValueError(f"Folder label tidak ada di daftar target: {unknown}")
y_ids = le.transform(train_labels)

# ---------- Build mask pairer & coverage ----------
mask_pairer = None
if USE_MASK:
    mask_pairer = MaskPairer(MASK_ROOT, phash_hash_size=PHASH_HASH_SIZE, phash_max_dist=PHASH_MAX_DIST)
    print(f"Mask STEM index size: {len(mask_pairer.stem2mask)}")
    mask_pairer.coverage_report(train_paths, base_root=train_dir, max_show=5)

    # Tampilkan contoh pasangan & nilai piksel 3 sampel
    print_mask_pair_samples(train_paths, base_root=train_dir, pairer=mask_pairer, k=DEBUG_MASK_SAMPLES)
    shown=0
    print("Sampel nilai piksel mask (cek threshold):")
    for fp in train_paths:
        mp, how = mask_pairer.find_for_train_rgb(fp, train_dir)
        if mp:
            try:
                m = np.asarray(Image.open(mp).convert("L"))
                vals = np.unique(m)
                print(f" - {os.path.relpath(fp, train_dir)}  ({how}) → uniq={vals[:10].tolist()} (min={int(vals.min())}, max={int(vals.max())})")
                shown += 1
                if shown>=3: break
            except Exception:
                pass
    if shown==0:
        print("   (belum ada pasangan yang terbaca)")
else:
    print("Mask: DISABLED")

# ---------- Split ----------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, va_idx = next(iter(skf.split(train_paths, y_ids)))
tr_items = [train_paths[i] for i in tr_idx]
va_items = [train_paths[i] for i in va_idx]
tr_lbls  = [y_ids[i] for i in tr_idx]
va_lbls  = [y_ids[i] for i in va_idx]

# (opsional) Cek konsistensi mask vs label
if USE_MASK:
    check_mask_label_consistency(train_paths, y_ids, train_dir, mask_pairer, max_show=10)

# ---------- Dataset ----------
class ImgDataset(Dataset):
    def __init__(self, items, labels=None, augment=False, base_root=None,
                 mask_mode="crop", mask_pairer: MaskPairer=None):
        self.items   = list(items)
        self.labels  = None if labels is None else list(labels)
        self.augment = bool(augment)
        self.base_root = base_root
        self.mask_mode = mask_mode
        self.mask_pairer = mask_pairer

    def __len__(self): 
        return len(self.items)

    def _apply_mask_mode(self, img_rgb: Image.Image, mask_img: Image.Image, mode: str, y_id: int|None):
        # Samakan ukuran
        if mask_img.size != img_rgb.size:
            mask_img = mask_img.resize(img_rgb.size, Image.NEAREST)

        # Ambil sebagai label map 0..255
        m = np.asarray(mask_img.convert("L"), dtype=np.uint8)

        # Validasi nilai di luar skema → clamp ke 0..15
        uniq = np.unique(m)
        if not set(uniq.tolist()).issubset(MASK_ALLOWED_VALUES):
            m = np.where(m > 15, 0, m).astype(np.uint8)

        # ==== Strategi binarisasi ====
        if MASK_BIN_STRATEGY == "target" and (y_id is not None):
            # pixel value benar untuk kelas ini = label_id + 1
            cls_val = int(y_id) + 1
            mask_bin = (m == cls_val)

        elif MASK_BIN_STRATEGY == "majority":
            vals = m[m > 0]
            if vals.size == 0:
                mask_bin = (m > 0)
            else:
                u, c = np.unique(vals, return_counts=True)
                maj_val = int(u[np.argmax(c)])
                mask_bin = (m == maj_val)

        else:  # "any"
            mask_bin = (m > 0)

        # noise cleaning kecil
        mask_img = Image.fromarray(mask_bin.astype(np.uint8)*255).filter(ImageFilter.MaxFilter(3)).filter(ImageFilter.MinFilter(3))
        mask_bin = np.asarray(mask_img, dtype=np.uint8) > 0

        # Kecil sekali? skip
        if mask_bin.sum() < MASK_MIN_PIX:
            return img_rgb, None

        if mode == "multiply":
            arr = np.asarray(img_rgb)
            if arr.ndim == 3:
                mask3 = mask_bin[..., None]
                arr = arr * mask3
            else:
                arr = arr * mask_bin
            return Image.fromarray(arr.astype(np.uint8)), None

        # mode == crop
        ys, xs = np.where(mask_bin)
        x0, y0, x1, y1 = xs.min(), ys.min(), xs.max(), ys.max()
        pad_x = int(MASK_PAD * img_rgb.width)
        pad_y = int(MASK_PAD * img_rgb.height)
        x0 = max(0, x0 - pad_x); y0 = max(0, y0 - pad_y)
        x1 = min(img_rgb.width - 1, x1 + pad_x); y1 = min(img_rgb.height - 1, y1 + pad_y)
        return img_rgb.crop((x0, y0, x1 + 1, y1 + 1)), None

    def __getitem__(self, idx):
        fp = self.items[idx]
        img_orig = Image.open(fp).convert("RGB")
        img_masked = None

        mp = None
        if self.mask_pairer:
            mp, _ = self.mask_pairer.find_for_train_rgb(fp, self.base_root or os.path.dirname(fp))

        if mp and os.path.exists(mp):
            try:
                mimg = Image.open(mp).convert("L")
                y_id = None if (self.labels is None) else int(self.labels[idx])

                if self.mask_mode == "dual":
                    img_masked, _ = self._apply_mask_mode(img_orig, mimg, mode="multiply", y_id=y_id)
                else:
                    img_processed, _ = self._apply_mask_mode(img_orig, mimg, mode=self.mask_mode, y_id=y_id)
                    img_orig = img_processed
            except Exception:
                pass

        # Pilih transform
        if self.augment:
            tfm = train_tfms_crop_safe if self.mask_mode=="crop" else train_tfms_default
        else:
            tfm = valid_tfms

        # dual-view: sinkron augment
        if self.mask_mode == "dual":
            if img_masked is None:
                img_masked = img_orig.copy()
            seed = torch.seed()
            torch.manual_seed(seed); img1 = tfm(img_orig)
            torch.manual_seed(seed); img2 = tfm(img_masked)
            proc1 = call_processor(img1); pv1 = {k: v.squeeze(0) for k, v in proc1.items()}
            proc2 = call_processor(img2); pv2 = {k: v.squeeze(0) for k, v in proc2.items()}
            pixel = (pv1, pv2)
        else:
            img_out = tfm(img_orig)
            proc = call_processor(img_out); pixel = {k: v.squeeze(0) for k, v in proc.items()}

        if self.labels is None:
            return pixel, os.path.basename(fp)
        else:
            return pixel, int(self.labels[idx])

def collate_fn_train(batch):
    first = batch[0][0]
    if isinstance(first, (tuple, list)):   # dual: (pv1, pv2)
        keys = first[0].keys()
        pv1 = {k: torch.stack([b[0][0][k] for b in batch], dim=0) for k in keys}
        pv2 = {k: torch.stack([b[0][1][k] for b in batch], dim=0) for k in keys}
        targets = torch.tensor([int(b[1]) for b in batch], dtype=torch.long)
        return (pv1, pv2), targets
    else:
        keys = first.keys()
        pixel_values = {k: torch.stack([b[0][k] for b in batch], dim=0) for k in keys}
        targets = torch.tensor([int(b[1]) for b in batch], dtype=torch.long)
        return pixel_values, targets

def collate_fn_test(batch):
    first = batch[0][0]
    if isinstance(first, (tuple, list)):
        keys = first[0].keys()
        pv1 = {k: torch.stack([b[0][0][k] for b in batch], dim=0) for k in keys}
        pv2 = {k: torch.stack([b[0][1][k] for b in batch], dim=0) for k in keys}
        names = [b[1] for b in batch]
        return (pv1, pv2), names
    else:
        keys = first.keys()
        pixel_values = {k: torch.stack([b[0][k] for b in batch], dim=0) for k in keys}
        names = [b[1] for b in batch]
        return pixel_values, names

pin_mem = torch.cuda.is_available()
ds_tr = ImgDataset(tr_items, tr_lbls, augment=True,
                   base_root=train_dir, mask_mode=MASK_MODE, mask_pairer=mask_pairer)
ds_va = ImgDataset(va_items, va_lbls, augment=False,
                   base_root=train_dir, mask_mode=MASK_MODE, mask_pairer=mask_pairer)
dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True,
                   num_workers=NUM_WORKERS, pin_memory=pin_mem, collate_fn=collate_fn_train)
dl_va = DataLoader(ds_va, batch_size=BATCH, shuffle=False,
                   num_workers=NUM_WORKERS, pin_memory=pin_mem, collate_fn=collate_fn_train)

# ---------- Feature dim (ROBUST) ----------
def _normalize_pv_for_probe(pv):
    # pv bisa dict, tuple(dict,dict), list[dict,dict]
    if isinstance(pv, dict):
        return pv
    if isinstance(pv, (tuple, list)) and len(pv) > 0 and isinstance(pv[0], dict):
        return pv[0]
    raise TypeError(f"Unexpected pixel_values type for probe: {type(pv)}")

with torch.no_grad():
    sample_inputs, _ = next(iter(dl_tr))
    sample_inputs = _normalize_pv_for_probe(sample_inputs)
    sample_inputs = {k: v.to(DEVICE) for k, v in sample_inputs.items()}
    feat_dim = backbone_features(sample_inputs).shape[-1]
print("\n=== FEATURES ===")
print("Feature dim:", feat_dim)

# ---------- Head ----------
head = nn.Sequential(
    nn.LayerNorm(feat_dim),
    nn.Linear(feat_dim, len(labels_ind))
).to(DEVICE)

opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

def run_epoch(loader, train=True):
    head.train() if train else head.eval()
    losses, preds, trues = [], [], []
    for pixel_values, y in loader:
        if isinstance(pixel_values, (tuple, list)):  # dual-view
            pv1, pv2 = pixel_values
            pv1 = {k: v.to(DEVICE, non_blocking=True) for k, v in pv1.items()}
            pv2 = {k: v.to(DEVICE, non_blocking=True) for k, v in pv2.items()}
        else:
            pixel_values = {k: v.to(DEVICE, non_blocking=True) for k, v in pixel_values.items()}
        y = y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.no_grad():
                if isinstance(pixel_values, (tuple, list)):
                    f1 = backbone_features(pv1)
                    f2 = backbone_features(pv2)
                    feats = (f1 + f2) / 2.0
                else:
                    feats = backbone_features(pixel_values)
            feats = feats.to(dtype=next(head.parameters()).dtype)
            logits = head(feats)
            loss = criterion(logits, y)

        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        losses.append(loss.item())
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        trues.extend(y.detach().cpu().tolist())
    return float(np.mean(losses)), accuracy_score(trues, preds)

# ---------- Train (EARLY STOPPING) ----------
best_val, best_state = 0.0, None
best_epoch = 0
no_improve_epochs = 0

print("\n=== TRAINING (EarlyStopping aktif) ===")
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = run_epoch(dl_tr, train=True)
    va_loss, va_acc = run_epoch(dl_va, train=False)
    print(f"[{epoch:02d}] train {tr_loss:.4f}/{tr_acc:.4f} | val {va_loss:.4f}/{va_acc:.4f}", end="")
    improved = (va_acc - best_val) >= EARLY_STOP_MIN_DELTA
    if improved or (epoch == 1 and va_acc > best_val):
        best_val = va_acc
        best_state = head.state_dict()
        best_epoch = epoch
        no_improve_epochs = 0
        print(f"  ↳ improve ≥ {EARLY_STOP_MIN_DELTA:.3f} → save best")
    else:
        no_improve_epochs += 1
        print(f"  ↳ no significant improve ({no_improve_epochs}/{EARLY_STOP_PATIENCE})")
    if no_improve_epochs >= EARLY_STOP_PATIENCE:
        print(f"\n⏹ Early stopping triggered at epoch {epoch}. Best epoch: {best_epoch} (val_acc={best_val:.4f})")
        break

if best_state is not None:
    head.load_state_dict(best_state)
    torch.save(head.state_dict(), "linear_head_fgclip2.pt")
    print(f"Saved best head -> linear_head_fgclip2.pt (best_epoch={best_epoch}, best_val_acc={best_val:.4f})")
else:
    print("⚠ Tidak ada state terbaik (cek loop training).")

# ---------- Inference (TEST) ----------
print("\n=== INFERENCE (TEST) ===")
test_paths = []
for img_id in range(1, 2058):
    base = f"{img_id:04d}"
    for ext in [".jpg", ".png", ".jpeg"]:
        p = os.path.join(test_dir, base + ext)
        if os.path.exists(p):
            test_paths.append(p); break
print(f"Test files : {len(test_paths)}")

# Test: single view (tanpa mask)
test_ds = ImgDataset(test_paths, labels=None, augment=False,
                     base_root=None, mask_mode="crop", mask_pairer=None)
test_dl = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=pin_mem, collate_fn=collate_fn_test)

pred_rows = []
head.eval()
with torch.no_grad():
    for pixel_values, basenames in tqdm(test_dl, total=len(test_dl)):
        if isinstance(pixel_values, (tuple, list)): pixel_values = pixel_values[0]
        pixel_values = {k: v.to(DEVICE, non_blocking=True) for k, v in pixel_values.items()}
        feats = backbone_features(pixel_values)
        feats = feats.to(dtype=next(head.parameters()).dtype)
        logits = head(feats)
        probs = torch.softmax(logits, dim=1)
        top_probs, top_idx = probs.max(dim=1)
        for bname, cls_id, conf in zip(basenames, top_idx.cpu().tolist(), top_probs.cpu().tolist()):
            try:
                img_id = int(Path(bname).stem)
            except:
                img_id = bname
            label = le.inverse_transform([cls_id])[0]
            pred_rows.append({"ID": img_id, "Label": label})
            if PRINT_ALL_TEST:
                print(f"ID: {img_id} | Pred: {label:<13s} | Confidence: {conf*100:.2f}%")

df = pd.DataFrame(pred_rows).sort_values("ID")
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Saved: {OUTPUT_CSV} | rows={len(df)} | best val acc={best_val:.3f}")

print("\n=== SUMMARY ===")
print(f"TRAIN_DIR : {train_dir}")
print(f"USE_MASK  : {USE_MASK}")
if USE_MASK:
    print(f"MASK_ROOT : {MASK_ROOT} (mode={MASK_MODE}, pad={MASK_PAD}, min_pix={MASK_MIN_PIX})")
    print(f"STEM index: {len(mask_pairer.stem2mask)} | RGB phash: {len(mask_pairer.rgb_phash)} | phash_max_dist={PHASH_MAX_DIST}")
    print(f"MASK_BIN  : {MASK_BIN_STRATEGY}")
print(f"MODEL_ID  : {MODEL_ID}")
print(f"OUTPUT    : {OUTPUT_CSV}")


=== RUNTIME ===
Python : 3.11.13
CUDA   : True
GPU    : Tesla T4
EarlyStopping: patience=5, min_delta=0.002


2025-11-11 12:30:01.404575: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762864201.544542      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762864201.584805      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

... (10 outputs left out)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



=== DATA CONFIG ===
TRAIN_DIR : /kaggle/input/v1-masakan-nusantara/action.v10i.folder/train
TEST_DIR  : /kaggle/input/data-mining-action-2025/test/test/
USE_MASK  : True
MASK_ROOT : /kaggle/input/mask-633-orient-auto-resize640-ref-contrast-adapt/train
MASK_MODE : dual | PAD=0.06 | MIN_PIX=64
PHASH     : size=8, max_dist=8
MASK_BIN  : target (target uses label_id+1)
Class distribution: Counter({'Bakso': 163, 'Sate Madura': 156, 'Rawon': 147, 'Nasi Goreng': 145, 'Ayam Goreng': 143, 'Coto Makassar': 143, 'Ayam Betutu': 137, 'Ayam Bakar': 127, 'Gado Gado': 127, 'Rendang': 124, 'Soto': 121, 'Pempek': 118, 'Ayam Pop': 108, 'Gudeg': 108, 'Sate Padang': 104})
Mask scan   : scanned=1266, indexed(mask-only)=633, overwritten_by_prefer=0
Sample mask indexed:
  - hzgnpg_jpg.rf.dda7c915caa0cb7ad16b50babb36bd3b_mask.png
  - z7dlb3_jpg.rf.c11d29e68bd95bbbd988c84fc404e23c_mask.png
  - 8bcpf9_jpg.rf.6e1b8675ea27b8ee014dba5549cb57af_mask.png
  - 31xkik_jpg.rf.e20ab3e76fb0065ed2da5f848f7ef2f5_mask.png
  

  0%|          | 0/33 [00:00<?, ?it/s]

ID: 1 | Pred: Sate Padang   | Confidence: 70.57%
ID: 2 | Pred: Rendang       | Confidence: 27.91%
ID: 3 | Pred: Ayam Betutu   | Confidence: 70.92%
ID: 4 | Pred: Rendang       | Confidence: 96.36%
ID: 5 | Pred: Ayam Bakar    | Confidence: 42.32%
ID: 6 | Pred: Coto Makassar | Confidence: 72.48%
ID: 7 | Pred: Ayam Bakar    | Confidence: 72.40%
ID: 8 | Pred: Gudeg         | Confidence: 97.77%
ID: 9 | Pred: Bakso         | Confidence: 99.04%
ID: 10 | Pred: Pempek        | Confidence: 98.83%
ID: 11 | Pred: Ayam Betutu   | Confidence: 97.58%
ID: 12 | Pred: Soto          | Confidence: 89.34%
ID: 13 | Pred: Sate Madura   | Confidence: 94.32%
ID: 14 | Pred: Pempek        | Confidence: 84.23%
ID: 15 | Pred: Coto Makassar | Confidence: 86.75%
ID: 16 | Pred: Ayam Goreng   | Confidence: 92.11%
ID: 17 | Pred: Bakso         | Confidence: 99.72%
ID: 18 | Pred: Ayam Goreng   | Confidence: 96.42%
ID: 19 | Pred: Ayam Goreng   | Confidence: 98.73%
ID: 20 | Pred: Pempek        | Confidence: 98.75%
ID: 21 | 

In [2]:
# ============================
# EVALUATION & REPORTING BLOCK
# ============================
import os, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_fscore_support,
    accuracy_score, top_k_accuracy_score, f1_score, balanced_accuracy_score,
    roc_auc_score, average_precision_score
)
from sklearn.preprocessing import label_binarize

OUTPUT_DIR = "metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

@torch.no_grad()
def infer_logits_and_targets(loader, device=DEVICE):
    head.eval()
    all_logits = []
    all_targets = []
    for pixel_values, y in loader:
        # Handle dual-view (mask_mode="dual") vs single-view
        if isinstance(pixel_values, (tuple, list)):
            pv1, pv2 = pixel_values
            pv1 = {k: v.to(device, non_blocking=True) for k, v in pv1.items()}
            pv2 = {k: v.to(device, non_blocking=True) for k, v in pv2.items()}
            f1 = backbone_features(pv1)
            f2 = backbone_features(pv2)
            feats = (f1 + f2) / 2.0
        else:
            pixel_values = {k: v.to(device, non_blocking=True) for k, v in pixel_values.items()}
            feats = backbone_features(pixel_values)

        feats = feats.to(dtype=next(head.parameters()).dtype)
        logits = head(feats)
        all_logits.append(logits.detach().cpu())
        all_targets.extend(y.detach().cpu().tolist())

    logits = torch.cat(all_logits, dim=0)
    y_true = np.asarray(all_targets, dtype=int)
    probs = torch.softmax(logits, dim=1).numpy()
    y_pred = logits.argmax(dim=1).numpy()
    return y_true, y_pred, probs

# 1) Inference pada validation set
y_true, y_pred, probs = infer_logits_and_targets(dl_va, device=DEVICE)
n_classes = len(labels_ind)

# 2) Metrik global
acc_top1   = accuracy_score(y_true, y_pred)
acc_top5   = top_k_accuracy_score(y_true, probs, k=min(5, n_classes), labels=np.arange(n_classes))
f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1_weight  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
bal_acc    = balanced_accuracy_score(y_true, y_pred)

# 3) Metrik per-kelas
prec, rec, f1, sup = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(n_classes), zero_division=0
)
per_class_df = pd.DataFrame({
    "class": labels_ind,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "support": sup
}).sort_values("class").reset_index(drop=True)

# 4) Classification report (string) + simpan
clf_rep = classification_report(
    y_true, y_pred, target_names=labels_ind, digits=4, zero_division=0
)
with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write("=== GLOBAL METRICS ===\n")
    f.write(f"Top-1 Accuracy : {acc_top1:.4f}\n")
    f.write(f"Top-5 Accuracy : {acc_top5:.4f}\n")
    f.write(f"Macro F1       : {f1_macro:.4f}\n")
    f.write(f"Weighted F1    : {f1_weight:.4f}\n")
    f.write(f"Balanced Acc   : {bal_acc:.4f}\n\n")
    f.write("=== CLASSIFICATION REPORT ===\n")
    f.write(clf_rep)

# 5) Confusion Matrix (raw & normalized)
cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
cm_norm = cm.astype(np.float64) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

# Simpan sebagai CSV (berguna untuk tabel di paper)
pd.DataFrame(cm, index=labels_ind, columns=labels_ind).to_csv(
    os.path.join(OUTPUT_DIR, "confusion_matrix_raw.csv")
)
pd.DataFrame(cm_norm, index=labels_ind, columns=labels_ind).to_csv(
    os.path.join(OUTPUT_DIR, "confusion_matrix_normalized.csv")
)

# Plot Confusion Matrix (normalized)
plt.figure(figsize=(10, 8), dpi=150)
plt.imshow(cm_norm, interpolation="nearest")
plt.title("Confusion Matrix (Normalized)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(ticks=np.arange(n_classes), labels=labels_ind, rotation=45, ha="right")
plt.yticks(ticks=np.arange(n_classes), labels=labels_ind)
plt.colorbar(fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_normalized.png"))
plt.close()

# 6) ROC-AUC (OvR) & Average Precision (PR-AUC) — opsional untuk multi-class
#    Catatan: butuh semua kelas muncul di y_true. Jika fold tertentu tidak lengkap, blok ini akan fallback.
try:
    y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
    roc_auc_macro = roc_auc_score(y_true_bin, probs, average="macro", multi_class="ovr")
    ap_macro      = average_precision_score(y_true_bin, probs, average="macro")
except Exception as e:
    roc_auc_macro = np.nan
    ap_macro      = np.nan

# 7) Ringkasan global → CSV
summary_df = pd.DataFrame([{
    "top1_acc": acc_top1,
    "top5_acc": acc_top5,
    "macro_f1": f1_macro,
    "weighted_f1": f1_weight,
    "balanced_acc": bal_acc,
    "macro_roc_auc_ovr": roc_auc_macro,
    "macro_avg_precision": ap_macro,
    "val_samples": int(len(y_true))
}])
summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)

# 8) Simpan per-kelas → CSV (siap dipakai jadi tabel di makalah)
per_class_df.to_csv(os.path.join(OUTPUT_DIR, "per_class_metrics.csv"), index=False)

# 9) Cetak ringkas ke terminal
print("\n=== VALIDATION METRICS ===")
print(f"Top-1 Acc      : {acc_top1:.4f}")
print(f"Top-5 Acc      : {acc_top5:.4f}")
print(f"Macro F1       : {f1_macro:.4f}")
print(f"Weighted F1    : {f1_weight:.4f}")
print(f"Balanced Acc   : {bal_acc:.4f}")
print(f"Macro ROC-AUC  : {roc_auc_macro:.4f}" if not np.isnan(roc_auc_macro) else "Macro ROC-AUC  : NA")
print(f"Macro AP (PR)  : {ap_macro:.4f}"      if not np.isnan(ap_macro) else "Macro AP (PR)  : NA")
print("\n=== PER-CLASS (head) ===")
print(per_class_df.head())
print(f"\nFiles saved → {OUTPUT_DIR}/")



=== VALIDATION METRICS ===
Top-1 Acc      : 0.8987
Top-5 Acc      : 0.9949
Macro F1       : 0.8986
Weighted F1    : 0.8977
Balanced Acc   : 0.9028
Macro ROC-AUC  : 0.9958
Macro AP (PR)  : 0.9624

=== PER-CLASS (head) ===
         class  precision    recall        f1  support
0   Ayam Bakar   0.833333  0.961538  0.892857       26
1  Ayam Betutu   0.821429  0.851852  0.836364       27
2  Ayam Goreng   0.913043  0.724138  0.807692       29
3     Ayam Pop   0.950000  0.904762  0.926829       21
4        Bakso   0.911765  0.939394  0.925373       33

Files saved → metrics/


In [4]:
!zip -r /kaggle/working/metrics.zip /kaggle/working/metrics


  adding: kaggle/working/metrics/ (stored 0%)
  adding: kaggle/working/metrics/per_class_metrics.csv (deflated 54%)
  adding: kaggle/working/metrics/confusion_matrix_raw.csv (deflated 68%)
  adding: kaggle/working/metrics/confusion_matrix_normalized.csv (deflated 77%)
  adding: kaggle/working/metrics/summary_metrics.csv (deflated 32%)
  adding: kaggle/working/metrics/confusion_matrix_normalized.png (deflated 19%)
  adding: kaggle/working/metrics/classification_report.txt (deflated 60%)


Menyimpan Model 

In [5]:
import torch, joblib, os
os.makedirs("/kaggle/working", exist_ok=True)

# simpan head (PyTorch)
torch.save({
    "feat_dim": int(feat_dim),
    "labels_ind": list(labels_ind),
    "model_id": MODEL_ID,
    "img_size": int(IMG_SIZE),
    "state_dict": head.state_dict(),
}, "/kaggle/working/linear_probe_bundle.pth")

# simpan LabelEncoder (scikit-learn)
joblib.dump(le, "/kaggle/working/label_encoder.joblib", compress=3)


['/kaggle/working/label_encoder.joblib']

Load Model

In [ ]:
import torch, joblib
from torch import nn

ckpt = torch.load("/kaggle/working/linear_probe_bundle.pth", map_location="cpu")
feat_dim    = ckpt["feat_dim"]
labels_ind  = ckpt["labels_ind"]

head = nn.Sequential(
    nn.LayerNorm(feat_dim),
    nn.Linear(feat_dim, len(labels_ind))
)
head.load_state_dict(ckpt["state_dict"])
head.eval()

le = joblib.load("/kaggle/working/label_encoder.joblib")


Inference 

In [ ]:
from transformers import AutoModel, AutoImageProcessor
import torch, joblib
from torch import nn

# 1️⃣ Memuat backbone pretrained 
MODEL_ID = "google/siglip2-giant-opt-patch16-384"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(device)
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model.eval()  # backbone tetap beku

# 2️⃣ Memuat linear head 
ckpt = torch.load("/kaggle/working/linear_probe_bundle.pth", map_location=device)
feat_dim   = ckpt["feat_dim"]
labels_ind = ckpt["labels_ind"]

head = nn.Sequential(
    nn.LayerNorm(feat_dim),
    nn.Linear(feat_dim, len(labels_ind))
).to(device)
head.load_state_dict(ckpt["state_dict"])
head.eval()

# 3️⃣ Memuat label encoder
le = joblib.load("/kaggle/working/label_encoder.joblib")

# 4️⃣ Menggunakan untuk inferensi
from PIL import Image
img = Image.open("contoh.jpg").convert("RGB")
inputs = image_processor(images=img, return_tensors="pt").to(device)

with torch.no_grad():
    feats = model.get_image_features(**inputs)
    logits = head(feats)
    pred = logits.argmax(1).item()
    label = le.inverse_transform([pred])[0]

print("Prediksi:", label)
